In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
df = pd.read_parquet(
    "s3://eagle-eye-ai-data/processed-data/candidate_risk_dataset/"
)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df["total_outstanding_debt"] = (
    df["total_outstanding_debt"]
    .fillna(0)
    .clip(lower=0)
)

df["total_outstanding_debt"] = np.log1p(
    df["total_outstanding_debt"]
)

In [ ]:
df["total_outstanding_debt"].describe()

In [ ]:
df["total_outstanding_debt"].describe()

In [ ]:
X = df.drop(columns=["target", "sk_id_curr"])
y = df["target"]

In [ ]:
print(X.head());

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=y.value_counts()[0] / y.value_counts()[1],
    eval_metric="logloss"
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
risk_scores = model.predict_proba(X_test)[:, 1]

In [ ]:
results = pd.DataFrame({
    "sk_id_curr": df.loc[X_test.index, "sk_id_curr"],
    "risk_score": risk_scores
})

In [ ]:
results.head(10)

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

In [ ]:
results["risk_level"] = pd.cut(
    results["risk_score"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"]
)

results.head(10)
results["risk_level"].value_counts()

In [ ]:
!pip install boto3

In [ ]:
import boto3

dynamodb = boto3.resource("dynamodb")

table = dynamodb.Table("CustomerRiskScores_candidate")

In [ ]:
import boto3

dynamodb = boto3.resource("dynamodb")

table = dynamodb.Table("CustomerRiskScores_candidate")

print(table.table_status)

In [ ]:

response = table.scan(
    Select='COUNT'
)

print(response['Count'])

In [ ]:
from decimal import Decimal

table.put_item(
    Item={
        "sk_id_curr": 999999,
        "risk_score": Decimal("0.95"),
        "risk_level": "High"
    }
)

In [ ]:
results.to_parquet(
    "risk_scores.parquet",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

results["risk_level"].value_counts().plot(kind="bar")

plt.title("Customer Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Number of Customers")
plt.show()
plt.savefig("cutomer_risk_level_distribution.png")

In [ ]:
import matplotlib.pyplot as plt

plt.hist(results["risk_score"], bins=20)

plt.title("Distribution of Risk Scores")
plt.xlabel("Risk Score")
plt.ylabel("Number of Customers")
plt.show()
plt.savefig("risk_distribution.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.Series({
    "avg_credit_utilization": 0.251112,
    "avg_delay": 0.203702,
    "active_loans": 0.203343,
    "max_delay": 0.124383,
    "total_outstanding_debt": 0.116545,
    "late_payment_count": 0.100916
})

feature_importance.sort_values().plot(kind="barh")

plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()
plt.savefig("risk_factors.png")

In [ ]:
top10 = results.sort_values(
    "risk_score",
    ascending=False
).head(10)

print(top10)

In [ ]:
results.to_csv(
    "risk_scores.csv",
    index=False
)